In [1]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, classification_report
import numpy as np
import pandas as pd
import joblib

In [2]:
X_train_others = pd.read_parquet('X_train_others.parquet')
y_train_others = pd.read_parquet('y_train_others.parquet')
X_train_others_smote = pd.read_parquet('X_train_others_smote.parquet')
y_train_others_smote = pd.read_parquet('y_train_others_smote.parquet')
X_val_others = pd.read_parquet('X_val_others.parquet')
y_val_others = pd.read_parquet('y_val_others.parquet')
X_test_others = pd.read_parquet('X_test_others.parquet')
y_test_others = pd.read_parquet('y_test_others.parquet')

In [3]:
X_train_others.shape

(196806, 218)

### Methodology
First train on non-SMOTE data. GridSearchCV will be used 

In [4]:
param_grid = {
    "criterion": ["gini", "entropy", "log_loss"],
    "splitter": ["best", "random"],
    "max_depth": [5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 5, 10],
    "max_features": [None, "sqrt"],
    "class_weight": [None, "balanced"],
    "ccp_alpha": [0.0, 0.001, 0.01]
}

In [5]:
dt = DecisionTreeClassifier(random_state=42)

grid_dt = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=2,
    refit=True
)

grid_dt.fit(X_train_others, y_train_others)

print("Best parameters:")
print(grid_dt.best_params_)

print("\nBest CV F1-score:")
print(grid_dt.best_score_)

Fitting 3 folds for each of 1944 candidates, totalling 5832 fits
Best parameters:
{'ccp_alpha': 0.001, 'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 10, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'splitter': 'best'}

Best CV F1-score:
0.2421963460353521


In [6]:
joblib.dump(grid_dt.best_estimator_, "models/decision_tree_best.pkl")

['models/decision_tree_best.pkl']

Determine the best threshold

In [7]:
grid_dt = joblib.load("models/decision_tree_best.pkl")

In [8]:
print(grid_dt.n_features_in_)
print(len(grid_dt.feature_importances_))

218
218


In [9]:
import hashlib

cols_hash = hashlib.md5(
    ",".join(X_train_others.columns).encode()
).hexdigest()

print(cols_hash)
print(X_train_others.shape)

51a839bbebe9a4cf07f0d44bae91bb6a
(196806, 218)


In [10]:
print(grid_dt.n_features_in_)
print(len(grid_dt.feature_names_in_))

218
218


In [11]:
missing = set(grid_dt.feature_names_in_) - set(X_train_others.columns)
extra = set(X_train_others.columns) - set(grid_dt.feature_names_in_)

print("In model but not dataframe:", missing)
print("In dataframe but not model:", extra)

In model but not dataframe: set()
In dataframe but not model: set()


In [12]:
print(len(missing), len(extra))

0 0


In [13]:
val_prob = grid_dt.predict_proba(X_val_others)[:, 1]

thresholds = np.arange(0.05, 1.00, 0.05)

best_threshold = 0.5
best_f1 = 0

for t in thresholds:
    pred = (val_prob >= t).astype(int)
    score = f1_score(y_val_others, pred)

    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("Best threshold:", best_threshold)
print("Validation F1:", best_f1)

Best threshold: 0.6000000000000001
Validation F1: 0.25168006324944


Test on the testing dataset

In [14]:
test_prob = grid_dt.predict_proba(X_test_others)[:, 1]

test_pred = (test_prob >= best_threshold).astype(int)

print(classification_report(y_test_others, test_pred))

              precision    recall  f1-score   support

           0       0.95      0.80      0.87     56538
           1       0.18      0.49      0.26      4965

    accuracy                           0.77     61503
   macro avg       0.56      0.64      0.56     61503
weighted avg       0.88      0.77      0.82     61503



### Train on SMOTE data
Now do the same for smote-enhanced data (note: only class_weight = balanced will be dropped since smote already balances classes)

In [15]:
param_grid_sm = {
    "criterion": ["gini", "entropy", "log_loss"],
    "splitter": ["best", "random"],
    "max_depth": [3, 10, 20, None],
    "min_samples_split": [2, 10, 30],
    "min_samples_leaf": [1, 5, 10],
    "max_features": [None, "sqrt"],
    "ccp_alpha": [0.0, 0.001, 0.01]
}

In [16]:
dt_sm = DecisionTreeClassifier(random_state=42)

grid_dt_sm = GridSearchCV(
    estimator=dt_sm,
    param_grid=param_grid_sm,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=2,
    refit=True
)

grid_dt_sm.fit(X_train_others, y_train_others)

print("Best parameters:")
print(grid_dt_sm.best_params_)

print("\nBest CV F1-score:")
print(grid_dt_sm.best_score_)

Fitting 3 folds for each of 1296 candidates, totalling 3888 fits
Best parameters:
{'ccp_alpha': 0.0, 'criterion': 'entropy', 'max_depth': None, 'max_features': None, 'min_samples_leaf': 5, 'min_samples_split': 2, 'splitter': 'best'}

Best CV F1-score:
0.1511905934366148


In [17]:
joblib.dump(grid_dt_sm.best_estimator_, "models/decision_tree_smote_best.pkl")

['models/decision_tree_smote_best.pkl']

Determine the best threshold

In [18]:
val_prob = grid_dt_sm.predict_proba(X_val_others)[:, 1]

thresholds = np.arange(0.05, 1.00, 0.05)

best_threshold = 0.5
best_f1 = 0

for t in thresholds:
    pred = (val_prob >= t).astype(int)
    score = f1_score(y_val_others, pred)

    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("Best threshold:", best_threshold)
print("Validation F1:", best_f1)

Best threshold: 0.05
Validation F1: 0.18379645128891864


Test on the testing dataset

In [19]:
test_prob = grid_dt_sm.predict_proba(X_test_others)[:, 1]

test_pred = (test_prob >= best_threshold).astype(int)

print(classification_report(y_test_others, test_pred))

              precision    recall  f1-score   support

           0       0.93      0.85      0.89     56538
           1       0.14      0.27      0.18      4965

    accuracy                           0.80     61503
   macro avg       0.53      0.56      0.53     61503
weighted avg       0.87      0.80      0.83     61503

